# Pneumonia Detection from Chest X-Ray Images using Deep Learning

## Project Overview
Binary classification of chest X-ray images as **Normal** or **Pneumonia** using:
1. Custom CNN (4-block architecture, trained from scratch)
2. VGG16 Transfer Learning (2-phase: frozen base then fine-tuning)
3. Traditional ML baselines (Logistic Regression, SVM, Random Forest with hand-crafted features)

## Dataset
**Citation:** Kermany, D., Zhang, K. and Goldbaum, M. (2018) Labeled Optical Coherence Tomography (OCT) and Chest X-Ray Images for Classification. Mendeley Data, V2. doi: 10.17632/rscbjbr9sj.2.

- The dataset (~1.1 GB) is **downloaded automatically** when the notebook is run.
- No manual download is required.

---

## Target Platform

| Spec | Value |
|------|-------|
| **Azure VM** | Standard_NC4as_T4_v3 |
| **GPU** | NVIDIA Tesla T4 (16 GB VRAM) |
| **Python** | 3.10+ |
| **Framework** | TensorFlow 2.x |

---

## How to Run

### Prerequisites
- The root project folder **must** be named **AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning**.
  This is the default name when cloning from the Git repository:

      git clone https://github.com/AkankshCaimi/AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning.git
      cd AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning

### Step 1 --- Environment Setup
Ensure the following packages are available. The notebook will attempt to **auto-install** any missing packages, but you can also install them manually:

    pip install tensorflow numpy pandas matplotlib seaborn opencv-python-headless Pillow scikit-learn scikit-image mahotas tqdm shap

### Step 2 --- Run the Notebook
1. Open the notebook (.ipynb) in JupyterLab / Jupyter Notebook on the Azure VM.
2. Ensure you are inside the **AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning** directory.
3. Click **Run All** (or Kernel > Restart and Run All).
4. The notebook will automatically:
   - Verify and install dependencies (Section 0.1)
   - Download and extract the dataset programmatically (Section 0.5)
   - Preprocess data, train all models, evaluate, and generate all plots end-to-end.

### Step 3 --- View Outputs
- Trained models are saved to ./models/
- All plots, metrics CSVs, and figures are saved to ./results/

---

## Estimated Runtime

| Mode | EPOCHS_CUSTOM | EPOCHS_TRANSFER_P1 | EPOCHS_TRANSFER_P2 | Approx. Time (T4 GPU) |
|------|:---:|:---:|:---:|:---:|
| **Reproducibility (default)** | 2 | 2 | 2 | ~15 min |
| **Full training (original)** | 50 | 30 | 20 | ~1.5 hours |

---

## Switching Between Reproducibility and Full Training

The notebook ships with **reduced hyperparameters** for fast, reproducible runs.
All original values are preserved as comments in **Section 0.4 (Global Constants)**.

To run with **original (full) hyperparameters**, open Section 0.4 and swap the comments:

    # Current (Reproducibility mode):
    EPOCHS_CUSTOM = 2              # Reduced for reproducibility on Azure T4
    EPOCHS_TRANSFER_P1 = 2         # Reduced for reproducibility on Azure T4
    EPOCHS_TRANSFER_P2 = 2         # Reduced for reproducibility on Azure T4

    # To restore original values, replace the above with:
    EPOCHS_CUSTOM = 50             # Original: full training
    EPOCHS_TRANSFER_P1 = 30        # Original: full training
    EPOCHS_TRANSFER_P2 = 20        # Original: full training

Similarly, EarlyStopping and ReduceLROnPlateau patience values are reduced
throughout the notebook. Each instance has a comment indicating the original value, e.g.:

    patience=5,    # Reduced for reproducibility (original: 10)

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| **0** | Environment setup, GPU verification, global constants, dataset download |
| **1** | Dataset loading, organization, and 85/15 train/val split |
| **2** | Exploratory Data Analysis (class distribution, image sizes, pixel intensities, average images) |
| **3** | Preprocessing and data pipeline (augmentation, generators, class weights) |
| **4** | Custom CNN --- build, train, evaluate |
| **5** | VGG16 Transfer Learning --- 2-phase training, evaluate |
| **6** | Traditional ML baselines (HOG/Haralick/LBP features + LR, SVM, RF) with SHAP explainability |
| **7** | Comprehensive comparison of all models (metrics, ROC curves, confusion matrices, compute cost) |
| **8** | Grad-CAM interpretability for both CNN models |
| **9** | Conclusions and summary |

---

## Output Files

### Models (./models/)

| File | Description |
|------|-------------|
| custom_cnn_best.keras | Best Custom CNN checkpoint (by val_auc) |
| custom_cnn_final.keras | Final Custom CNN after training |
| transfer_p1_best.keras | Best VGG16 checkpoint after Phase 1 |
| transfer_finetuned_best.keras | Best VGG16 checkpoint after Phase 2 |
| transfer_final.keras | Final VGG16 model after both phases |

### Results (./results/)

| File | Description |
|------|-------------|
| full_comparison.csv | All models metrics in one table |
| custom_cnn_metrics.csv | Custom CNN test metrics |
| transfer_metrics.csv | VGG16 test metrics |
| class_distribution.png | Class balance across splits |
| sample_images.png | Example X-rays from each class |
| size_distribution.png | Image dimension distributions |
| intensity_distribution.png | Pixel intensity analysis |
| average_images.png | Mean image per class plus difference |
| augmented_samples.png | Augmented training examples |
| custom_cnn_training.png | Custom CNN training curves |
| transfer_training.png | VGG16 training curves (Phase 1 + 2) |
| cm_custom_cnn.png | Custom CNN confusion matrix |
| cm_transfer.png | VGG16 confusion matrix |
| cm_traditional_ml.png | Traditional ML confusion matrices |
| feature_importance.png | Random Forest feature group importance |
| shap_summary.png | SHAP global feature importance |
| shap_individual.png | SHAP per-prediction explanations |
| comparison_bars.png | All models metrics bar chart |
| roc_comparison.png | ROC curves for all models |
| training_time.png | Computational cost comparison |
| all_confusion_matrices.png | Side-by-side confusion matrices |
| gradcam_analysis.png | Grad-CAM visualizations (VGG16) |
| gradcam_comparison.png | Grad-CAM Custom CNN vs VGG16 |

---



---
## Section 0 — Environment Setup & GPU Verification
 
This section verifies the compute environment, installs dependencies, imports all required libraries, and sets global constants used throughout the notebook.

In [ ]:
# ── 0.1 Verify Dependencies ──
 
import sys
print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}\n")
packages_to_check = [
    ("tensorflow", "tensorflow"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("cv2", "opencv"),
    ("PIL", "Pillow"),
    ("skimage", "scikit-image"),
    ("mahotas", "mahotas"),
    ("tqdm", "tqdm"),
]
all_good = True
for import_name, display_name in packages_to_check:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "OK")
        print(f"  ✅ {display_name:20s} → {ver}")
    except ImportError:
        print(f"  ❌ {display_name:20s} → NOT FOUND")
        all_good = False
if all_good:
    print("\nAll packages verified successfully.")
else:
    print("\nSome packages missing. Attempting automatic installation...")
    import subprocess
    pip_names = {
        "tensorflow": "tensorflow",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "sklearn": "scikit-learn",
        "cv2": "opencv-python-headless",
        "PIL": "Pillow",
        "skimage": "scikit-image",
        "mahotas": "mahotas",
        "tqdm": "tqdm",
    }
    for import_name, display_name in packages_to_check:
        try:
            __import__(import_name)
        except ImportError:
            pkg = pip_names.get(import_name, display_name)
            print(f"  Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print("  Installation complete. If errors persist, restart the kernel and re-run.")
 

In [ ]:
# ── 0.2 Import Libraries ──
 
import os
import random
import time
import warnings
import gc
import shutil
from pathlib import Path
 
import numpy as np
import pandas as pd
 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
 
import cv2
from PIL import Image
 
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
 
from skimage.feature import hog, local_binary_pattern
import mahotas
 
from tqdm import tqdm
 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Disable XLA JIT compilation — required for T4 GPU compatibility
# XLA's MaxPool gradient kernel is not implemented for deterministic mode on T4
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0 --tf_xla_cpu_global_jit=false"

# Do NOT set these — they trigger XLA deterministic mode which crashes on T4:
# os.environ["PYTHONHASHSEED"] = "42"             # Can trigger deterministic mode
# os.environ["TF_DETERMINISTIC_OPS"] = "1"        # Causes MaxPool XLA error on T4
# os.environ["TF_CUDNN_DETERMINISTIC"] = "1"      # Causes MaxPool XLA error on T4

import tensorflow as tf

# Disable XLA JIT at TF level as well
tf.config.optimizer.set_jit(False)
from tensorflow import keras
from tensorflow.keras import layers, models, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.applications import VGG16
 
warnings.filterwarnings("ignore")
 
print(f"Python version  : {sys.version}")
print(f"TensorFlow ver  : {tf.__version__}")
print(f"NumPy version   : {np.__version__}")
print(f"Keras version   : {keras.__version__}")

In [ ]:
# ── 0.3 GPU Verification ──

import tensorflow as tf

print("=" * 60)
print("  DEVICE VERIFICATION")
print("=" * 60)

devices = tf.config.list_physical_devices()
print("  Available devices:")
for d in devices:
    print(f"    {d}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        print(f"\n  ✅ GPU detected: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    print(f"  Number of GPUs: {len(gpus)}")
    gpu_details = tf.config.experimental.get_device_details(gpus[0]) if gpus else {}
    gpu_name = gpu_details.get("device_name", "Unknown")
    print(f"  GPU name: {gpu_name}")
    if "T4" in gpu_name or "NVIDIA" in gpu_name.upper():
        print("  NVIDIA CUDA GPU will be used for training.")
    else:
        print("  GPU will be used for training.")
else:
    print("\n  ⚠ No GPU detected — using CPU.")
    print("  Training will be slower but functional.")

print("=" * 60)

In [ ]:
# ── 0.4 Global Constants ──

BASE_DATA_DIR = "./AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning/chest_xray"
SPLIT_DATA_DIR = "./AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning/chest_xray_split"  # May be overwritten in Section 0.5
RESULTS_DIR = "./AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning/results"
MODELS_DIR = "./AIDL-Pneumonia-Detection-from-Chest-XRay-Images-Using-Deep-Learning/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = 224
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
BATCH_SIZE = 32

SEED = 42

# ── Reproducibility mode: reduced epochs for fast "Run All" ──
# To restore original (full training) values, uncomment the lines below
# and comment out the reduced values.

# EPOCHS_CUSTOM = 50          # Original: full training
# EPOCHS_TRANSFER_P1 = 30     # Original: full training
# EPOCHS_TRANSFER_P2 = 20     # Original: full training

EPOCHS_CUSTOM = 2              # Reduced for reproducibility on Azure T4
EPOCHS_TRANSFER_P1 = 2         # Reduced for reproducibility on Azure T4
EPOCHS_TRANSFER_P2 = 2         # Reduced for reproducibility on Azure T4

# ── Reproducibility seeds ──
# Full GPU determinism (TF_DETERMINISTIC_OPS / enable_op_determinism) is
# intentionally NOT enabled because it triggers:
#   "UNIMPLEMENTED: GPU MaxPool gradient ops do not yet have a deterministic
#    XLA implementation" on NVIDIA T4 GPUs.
# The three seeds below provide practical reproducibility with only minor
# floating-point variation (~0.1-0.5%) between runs.

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Global constants set.")
print(f"  Image size       : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Random seed      : {SEED}")
print(f"  Epochs (Custom)  : {EPOCHS_CUSTOM}")
print(f"  Epochs (TL P1)   : {EPOCHS_TRANSFER_P1}")
print(f"  Epochs (TL P2)   : {EPOCHS_TRANSFER_P2}")
print(f"  GPU determinism  : DISABLED (T4 XLA compatibility)")

In [ ]:
# ── 0.5 Programmatic Dataset Download ──
import os
import urllib.request
import zipfile
from tqdm import tqdm
import subprocess
import shutil

DATASET_URL = "https://data.mendeley.com/public-files/datasets/rscbjbr9sj/files/f12eaf6d-6023-432f-acc9-80c9d7393433/file_downloaded"

# ── Detect fastest writable disk ──
# Azure VMs have a fast local NVMe SSD at /tmp which is much faster
# for extracting thousands of small files than the network-attached OS disk.
LOCAL_SSD_DIR = None
for candidate_dir in ["/tmp/dataset_cache", "/mnt/dataset_cache"]:
    try:
        os.makedirs(candidate_dir, exist_ok=True)
        test_file = os.path.join(candidate_dir, ".write_test")
        with open(test_file, 'w') as f:
            f.write("test")
        os.remove(test_file)
        LOCAL_SSD_DIR = candidate_dir
        print(f"  Fast disk found: {LOCAL_SSD_DIR}")
        break
    except (PermissionError, OSError):
        continue

if LOCAL_SSD_DIR is None:
    LOCAL_SSD_DIR = "."
    print(f"  No fast disk available. Using working directory.")

ZIP_PATH = os.path.join(LOCAL_SSD_DIR, "chest_xray.zip")
LOCAL_SSD_EXTRACT = os.path.join(LOCAL_SSD_DIR, "chest_xray_extract")

# Store split data on same fast disk for faster training I/O
SPLIT_DATA_DIR = os.path.join(LOCAL_SSD_DIR, "chest_xray_split")


def validate_dataset(base_dir):
    """Check that the dataset has all 4 required directories with images."""
    required = {
        "train/NORMAL": 1000,
        "train/PNEUMONIA": 3000,
        "test/NORMAL": 200,
        "test/PNEUMONIA": 300,
    }
    for subdir, min_count in required.items():
        full_path = os.path.join(base_dir, subdir)
        if not os.path.isdir(full_path):
            return False
        count = len([f for f in os.listdir(full_path)
                     if f.lower().endswith((".jpeg", ".jpg", ".png"))])
        if count < min_count:
            return False
    return True


# ── Check if existing data is valid, handle stale symlinks ──
need_download = True

if os.path.islink(BASE_DATA_DIR):
    if os.path.exists(os.readlink(BASE_DATA_DIR)) and validate_dataset(BASE_DATA_DIR):
        need_download = False
    else:
        os.unlink(BASE_DATA_DIR)
        if os.path.exists(LOCAL_SSD_EXTRACT):
            shutil.rmtree(LOCAL_SSD_EXTRACT)
elif os.path.isdir(BASE_DATA_DIR):
    if validate_dataset(BASE_DATA_DIR):
        need_download = False
    else:
        shutil.rmtree(BASE_DATA_DIR)

if not need_download:
    print(f"Dataset validated at {BASE_DATA_DIR}. Skipping download.")
else:
    print("Downloading Mendeley dataset (~1.1 GB)...")

    req = urllib.request.Request(
        DATASET_URL,
        headers={
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                          'AppleWebKit/537.36 (KHTML, like Gecko) '
                          'Chrome/120.0.0.0 Safari/537.36'
        },
    )

    with urllib.request.urlopen(req) as response:
        total_size = int(response.info().get('Content-Length', 0))
        with open(ZIP_PATH, 'wb') as out_file, tqdm(
            desc="Downloading", total=total_size,
            unit='B', unit_scale=True, unit_divisor=1024,
        ) as bar:
            while True:
                chunk = response.read(8192)
                if not chunk:
                    break
                out_file.write(chunk)
                bar.update(len(chunk))

    # ── Extract to fast disk ──
    print("\nExtracting dataset...")
    if os.path.exists(LOCAL_SSD_EXTRACT):
        shutil.rmtree(LOCAL_SSD_EXTRACT)
    os.makedirs(LOCAL_SSD_EXTRACT, exist_ok=True)

    extraction_done = False
    try:
        subprocess.run(['which', 'unzip'], check=True, capture_output=True)
        subprocess.run(
            ['unzip', '-q', '-o', ZIP_PATH, '-d', LOCAL_SSD_EXTRACT],
            check=True,
        )
        extraction_done = True
        print("  Extracted using system 'unzip'.")
    except (subprocess.CalledProcessError, FileNotFoundError):
        pass

    if not extraction_done:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_SSD_EXTRACT)
        print("  Extracted using Python zipfile.")

    os.remove(ZIP_PATH)

    # ── Find dataset folder (handles double-nesting from Mendeley zip) ──
    # The zip sometimes extracts as: chest_xray/chest_xray/train/
    # We need the deepest folder that has train/NORMAL, train/PNEUMONIA,
    # test/NORMAL, and test/PNEUMONIA as subdirectories.
    dataset_folder = None
    for root, dirs, files in os.walk(LOCAL_SSD_EXTRACT):
        if "train" in dirs and "test" in dirs:
            train_path = os.path.join(root, "train")
            test_path = os.path.join(root, "test")
            if (os.path.isdir(os.path.join(train_path, "NORMAL")) and
                os.path.isdir(os.path.join(train_path, "PNEUMONIA")) and
                os.path.isdir(os.path.join(test_path, "NORMAL")) and
                os.path.isdir(os.path.join(test_path, "PNEUMONIA"))):
                dataset_folder = root
                break

    if dataset_folder is None:
        raise FileNotFoundError(
            "Could not find valid dataset structure (train/test with NORMAL/PNEUMONIA) "
            "after extraction. Check the Mendeley download URL."
        )

    # ── Symlink to working directory for fast I/O ──
    if os.path.exists(BASE_DATA_DIR):
        if os.path.islink(BASE_DATA_DIR):
            os.unlink(BASE_DATA_DIR)
        else:
            shutil.rmtree(BASE_DATA_DIR)

    os.symlink(os.path.abspath(dataset_folder), BASE_DATA_DIR)
    print(f"  Symlinked: {BASE_DATA_DIR} -> {os.path.abspath(dataset_folder)}")

    # ── Validate ──
    assert validate_dataset(BASE_DATA_DIR), "Dataset validation failed after extraction."
    print("Dataset ready.")

---
## Section 1 — Dataset Loading & Organization
 
The original dataset contains train and test folders. We will:
1. Verify the existing folder structure and count images
2. Create a proper train/validation/test split (85/15 stratified from training data)
3. Keep the original test set completely untouched
 
**Design Decision:** The original dataset either lacks a validation set or has a very small one (~16 images). A 15% stratified split from training data ensures reliable model selection during training.

In [ ]:
# ── 1.1 Verify Original Dataset Structure ──
 
print("=" * 60)
print("  ORIGINAL DATASET STRUCTURE")
print("=" * 60)
 
for split in ["train", "test"]:
    split_path = os.path.join(BASE_DATA_DIR, split)
    if os.path.exists(split_path):
        for cls in ["NORMAL", "PNEUMONIA"]:
            cls_path = os.path.join(split_path, cls)
            if os.path.exists(cls_path):
                count = len([
                    f for f in os.listdir(cls_path)
                    if f.lower().endswith((".jpeg", ".jpg", ".png"))
                ])
                print(f"  {split}/{cls}: {count} images")
            else:
                print(f"  {split}/{cls}: NOT FOUND")
    else:
        print(f"  {split}/: NOT FOUND")
 
# Check if val folder exists
val_path = os.path.join(BASE_DATA_DIR, "val")
if os.path.exists(val_path):
    print()
    for cls in ["NORMAL", "PNEUMONIA"]:
        cls_path = os.path.join(val_path, cls)
        if os.path.exists(cls_path):
            count = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpeg", ".jpg", ".png"))
            ])
            print(f"  val/{cls}: {count} images")
    print("\n  NOTE: Original val set exists but is too small for reliable validation.")
    print("  Will merge with training data and re-split.")
 
print("=" * 60)

In [ ]:
# ── 1.2 Create Proper Train / Validation / Test Split ──
 
def reorganize_dataset(base_dir, output_dir, val_split=0.15, seed=42):
    """
    Merge all training-related images then create a stratified
    85/15 train/val split. Test set remains untouched.
    
    Uses symbolic links instead of file copies for speed.
    Symlinks are instant (no data duplication) and the original
    files are not modified, so this is safe.
    """
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"  Removed existing directory: {output_dir}")

    # Choose linking strategy: symlink (instant) > hardlink (fast) > copy (slow)
    def fast_link(src, dst):
        try:
            os.symlink(os.path.abspath(src), dst)
        except OSError:
            try:
                os.link(src, dst)
            except OSError:
                shutil.copy2(src, dst)

    for cls in ["NORMAL", "PNEUMONIA"]:
        all_image_paths = []

        # Gather from train folder
        train_cls = os.path.join(base_dir, "train", cls)
        if os.path.exists(train_cls):
            for f in os.listdir(train_cls):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    all_image_paths.append(os.path.join(train_cls, f))

        # Gather from val folder if it exists
        val_cls = os.path.join(base_dir, "val", cls)
        if os.path.exists(val_cls):
            for f in os.listdir(val_cls):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    all_image_paths.append(os.path.join(val_cls, f))

        # Stratified split
        train_paths, val_paths = train_test_split(
            all_image_paths, test_size=val_split, random_state=seed
        )

        # Link to new directories (instead of copy)
        for split_name, paths in [("train", train_paths), ("val", val_paths)]:
            dest_dir = os.path.join(output_dir, split_name, cls)
            os.makedirs(dest_dir, exist_ok=True)
            for src_path in paths:
                dst_path = os.path.join(dest_dir, os.path.basename(src_path))
                fast_link(src_path, dst_path)

        # Link test set unchanged
        test_src = os.path.join(base_dir, "test", cls)
        test_dest = os.path.join(output_dir, "test", cls)
        os.makedirs(test_dest, exist_ok=True)
        if os.path.exists(test_src):
            for f in os.listdir(test_src):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    src_path = os.path.join(test_src, f)
                    dst_path = os.path.join(test_dest, f)
                    fast_link(src_path, dst_path)

    print("  Dataset reorganization complete (using symlinks — instant).")
 
 
print("Reorganizing dataset with 85/15 train/val split...\n")
reorganize_dataset(BASE_DATA_DIR, SPLIT_DATA_DIR, val_split=0.15, seed=SEED)

In [ ]:
# ── 1.3 Verify New Split ──
 
print("=" * 60)
print("  REORGANIZED DATASET STRUCTURE")
print("=" * 60)
 
split_stats = {}
for split in ["train", "val", "test"]:
    split_stats[split] = {}
    for cls in ["NORMAL", "PNEUMONIA"]:
        cls_path = os.path.join(SPLIT_DATA_DIR, split, cls)
        count = len([
            f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))
        ])
        split_stats[split][cls] = count
        print(f"  {split:5s} / {cls:10s} : {count:5d} images")
    total = sum(split_stats[split].values())
    print(f"  {split:5s} / {'TOTAL':10s} : {total:5d} images")
    print()
 
print("=" * 60)

---
## Section 2 — Exploratory Data Analysis
 
Thorough EDA to understand the dataset characteristics. Every finding here directly informs a preprocessing or model design decision.
 
Key questions:
1. How imbalanced are the classes?
2. What are the image size distributions?
3. Do pixel intensity patterns differ between classes?
4. What do average images reveal about class differences?

In [ ]:
# ── 2.1 Class Distribution Visualization ──
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
for idx, split in enumerate(["train", "val", "test"]):
    normal_count = split_stats[split]["NORMAL"]
    pneumonia_count = split_stats[split]["PNEUMONIA"]
 
    bars = axes[idx].bar(
        ["Normal", "Pneumonia"],
        [normal_count, pneumonia_count],
        color=["#2ecc71", "#e74c3c"],
        edgecolor="black",
        linewidth=0.8,
    )
    axes[idx].set_title(f"{split.capitalize()} Set", fontsize=14, fontweight="bold")
    axes[idx].set_ylabel("Number of Images")
 
    for bar, count in zip(bars, [normal_count, pneumonia_count]):
        axes[idx].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20,
            str(count),
            ha="center",
            fontweight="bold",
            fontsize=11,
        )
 
    ratio = pneumonia_count / normal_count if normal_count > 0 else 0
    axes[idx].text(
        0.5, 0.95, f"Ratio P:N = {ratio:.2f}:1",
        transform=axes[idx].transAxes,
        ha="center", fontsize=10, style="italic",
    )
 
plt.suptitle("Class Distribution Across Splits", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
 
train_ratio = split_stats["train"]["PNEUMONIA"] / split_stats["train"]["NORMAL"]
print(f"\nTraining set imbalance ratio (Pneumonia:Normal) = {train_ratio:.2f}:1")
print("Design Decision: This imbalance will be addressed using computed class weights.")

In [ ]:
# ── 2.2 Sample Image Visualization ──
 
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
 
for row, cls in enumerate(["NORMAL", "PNEUMONIA"]):
    cls_path = os.path.join(SPLIT_DATA_DIR, "train", cls)
    all_imgs = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    sample_images = random.sample(all_imgs, 6)
 
    for col, img_name in enumerate(sample_images):
        img = Image.open(os.path.join(cls_path, img_name))
        axes[row][col].imshow(img, cmap="gray")
        axes[row][col].set_title(cls, fontsize=10)
        axes[row][col].axis("off")
 
        w, h = img.size
        axes[row][col].text(
            5, 20, f"{w}x{h}", fontsize=7, color="yellow",
            bbox=dict(boxstyle="round", facecolor="black", alpha=0.7),
        )
 
plt.suptitle("Sample Chest X-Ray Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2.3 Image Size Distribution ──
 
widths, heights, classes_list = [], [], []
 
for cls in ["NORMAL", "PNEUMONIA"]:
    cls_path = os.path.join(SPLIT_DATA_DIR, "train", cls)
    for img_name in os.listdir(cls_path):
        if img_name.lower().endswith((".jpeg", ".jpg", ".png")):
            img = Image.open(os.path.join(cls_path, img_name))
            w, h = img.size
            widths.append(w)
            heights.append(h)
            classes_list.append(cls)
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
axes[0].hist(widths, bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(np.mean(widths), color="red", linestyle="--",
                label=f"Mean: {np.mean(widths):.0f}")
axes[0].set_title("Width Distribution", fontsize=13)
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Count")
axes[0].legend()
 
axes[1].hist(heights, bins=40, color="coral", edgecolor="black", alpha=0.8)
axes[1].axvline(np.mean(heights), color="red", linestyle="--",
                label=f"Mean: {np.mean(heights):.0f}")
axes[1].set_title("Height Distribution", fontsize=13)
axes[1].set_xlabel("Height (pixels)")
axes[1].legend()
 
for cls, color in [("NORMAL", "#2ecc71"), ("PNEUMONIA", "#e74c3c")]:
    mask = [c == cls for c in classes_list]
    w_cls = np.array(widths)[mask]
    h_cls = np.array(heights)[mask]
    axes[2].scatter(w_cls, h_cls, alpha=0.3, s=10, color=color, label=cls)
 
axes[2].set_title("Width vs Height", fontsize=13)
axes[2].set_xlabel("Width")
axes[2].set_ylabel("Height")
axes[2].legend()
 
plt.suptitle("Image Size Distribution", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "size_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
 
print(f"Width  — Min: {min(widths)}, Max: {max(widths)}, "
      f"Mean: {np.mean(widths):.0f}, Std: {np.std(widths):.0f}")
print(f"Height — Min: {min(heights)}, Max: {max(heights)}, "
      f"Mean: {np.mean(heights):.0f}, Std: {np.std(heights):.0f}")
print(f"\nDesign Decision: Variable sizes confirm need to resize all images to {IMG_SIZE}x{IMG_SIZE}.")

In [ ]:
# ── 2.4 Pixel Intensity Distribution ──
 
n_samples = 300
 
normal_means, pneumonia_means = [], []
normal_stds, pneumonia_stds = [], []
 
for cls, means_list, stds_list in [
    ("NORMAL", normal_means, normal_stds),
    ("PNEUMONIA", pneumonia_means, pneumonia_stds),
]:
    cls_path = os.path.join(SPLIT_DATA_DIR, "train", cls)
    all_imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    sampled = random.sample(all_imgs, min(n_samples, len(all_imgs)))
 
    for img_name in sampled:
        img = np.array(Image.open(os.path.join(cls_path, img_name)).convert("L"))
        means_list.append(img.mean())
        stds_list.append(img.std())
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
axes[0].hist(normal_means, bins=30, alpha=0.7, color="#2ecc71",
             label="Normal", edgecolor="black")
axes[0].hist(pneumonia_means, bins=30, alpha=0.7, color="#e74c3c",
             label="Pneumonia", edgecolor="black")
axes[0].set_title("Mean Pixel Intensity", fontsize=13)
axes[0].set_xlabel("Mean Pixel Value (0-255)")
axes[0].set_ylabel("Count")
axes[0].legend()
 
axes[1].hist(normal_stds, bins=30, alpha=0.7, color="#2ecc71",
             label="Normal", edgecolor="black")
axes[1].hist(pneumonia_stds, bins=30, alpha=0.7, color="#e74c3c",
             label="Pneumonia", edgecolor="black")
axes[1].set_title("Pixel Intensity Std Dev", fontsize=13)
axes[1].set_xlabel("Std Dev of Pixel Values")
axes[1].legend()
 
plt.suptitle("Pixel Intensity Analysis: Normal vs Pneumonia",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "intensity_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
 
print(f"Normal    — Mean intensity: {np.mean(normal_means):.1f} +/- {np.std(normal_means):.1f}")
print(f"Pneumonia — Mean intensity: {np.mean(pneumonia_means):.1f} +/- {np.std(pneumonia_means):.1f}")
print("\nObservation: Overlapping distributions suggest simple thresholding")
print("is insufficient — justifies the need for learned feature extraction (CNN).")

In [ ]:
# ── 2.5 Average Image per Class ──
 
def compute_average_image(cls_path, img_size=224, max_images=500):
    """Compute the mean image for a class."""
    all_imgs = [f for f in os.listdir(cls_path)
                if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    sampled = random.sample(all_imgs, min(max_images, len(all_imgs)))
 
    accumulator = np.zeros((img_size, img_size), dtype=np.float64)
    count = 0
    for img_name in sampled:
        img = cv2.imread(os.path.join(cls_path, img_name), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img_resized = cv2.resize(img, (img_size, img_size))
            accumulator += img_resized.astype(np.float64)
            count += 1
    return (accumulator / count).astype(np.uint8) if count > 0 else None
 
 
avg_normal = compute_average_image(os.path.join(SPLIT_DATA_DIR, "train", "NORMAL"))
avg_pneumonia = compute_average_image(os.path.join(SPLIT_DATA_DIR, "train", "PNEUMONIA"))
diff_image = cv2.absdiff(avg_normal, avg_pneumonia)
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
axes[0].imshow(avg_normal, cmap="gray")
axes[0].set_title("Average NORMAL", fontsize=13)
axes[0].axis("off")
 
axes[1].imshow(avg_pneumonia, cmap="gray")
axes[1].set_title("Average PNEUMONIA", fontsize=13)
axes[1].axis("off")
 
axes[2].imshow(diff_image, cmap="hot")
axes[2].set_title("Absolute Difference", fontsize=13)
axes[2].axis("off")
 
plt.suptitle("Average Image Analysis", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "average_images.png"), dpi=150, bbox_inches="tight")
plt.show()
 
print("The difference image highlights regions where Normal and Pneumonia")
print("X-rays diverge most — primarily in the lower lung fields, consistent")
print("with common pneumonia presentation patterns.")

### EDA Summary & Design Decisions
 
| Finding | Implication | Design Decision |
|---|---|---|
| Class imbalance (~2.9:1 Pneumonia:Normal) | Model may be biased toward majority class | Use computed class weights during training |
| Original val set too small (~16 images) | Unreliable validation metrics | Re-split into 85/15 stratified train/val |
| Variable image sizes (wide range) | Cannot feed directly to CNN | Resize all images to 224×224 |
| Overlapping pixel intensity distributions | Simple thresholding insufficient | Justifies use of deep feature learning (CNN) |
| Visible pattern differences in average images | Spatial patterns exist between classes | CNN architecture should capture spatial hierarchies |

---
## Section 3 — Preprocessing & Data Pipeline

### Design Decisions:
- **Resize to 224×224**: Standardizes variable-sized inputs; compatible with VGG16 for transfer learning
- **Normalize to [0,1]**: Rescale pixel values for stable gradient computation
- **Augmentation (training only)**: Clinically appropriate transformations to reduce overfitting
- **Class weights**: Computed from training distribution (2.88:1) to handle imbalance

### Augmentation Rationale:
| Augmentation | Included? | Clinical Justification |
|---|---|---|
| Rotation (±15°) | ✅ | Patient positioning varies between scans |
| Width/Height shift (10%) | ✅ | Slight centering variations in X-ray machines |
| Zoom (±10%) | ✅ | Varying patient-to-detector distances |
| Horizontal flip | ✅ | Anatomically valid for chest X-rays |
| Brightness (±15%) | ✅ | Different X-ray machine calibrations |
| Vertical flip | ❌ | Anatomically unrealistic — humans are not upside down |
| Heavy shear | ❌ | Would distort lung anatomy unnaturally |


In [ ]:
# ── 3.1 Create Data Generators ──

# Training: WITH clinically appropriate augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode="nearest",
)

# Validation & Test: Only rescaling — no augmentation
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

print("Data generators created.")
print("  Training   : augmentation ON")
print("  Validation : augmentation OFF")
print("  Test       : augmentation OFF")

In [ ]:
# ── 3.2 Create Directory Iterators ──

train_generator = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DATA_DIR, "train"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    seed=SEED,
    shuffle=True,
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DATA_DIR, "val"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    seed=SEED,
    shuffle=False,
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DATA_DIR, "test"),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    seed=SEED,
    shuffle=False,
)

print(f"\nClass mapping: {train_generator.class_indices}")
print(f"Training batches   : {len(train_generator)}")
print(f"Validation batches : {len(val_generator)}")
print(f"Test batches       : {len(test_generator)}")

In [ ]:
# ── 3.3 Visualize Augmented Images ──

# Get a batch of augmented training images
sample_batch, sample_labels = next(train_generator)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
class_names = ["NORMAL", "PNEUMONIA"]

for i in range(10):
    row = i // 5
    col = i % 5
    axes[row][col].imshow(sample_batch[i])
    label = class_names[int(sample_labels[i])]
    axes[row][col].set_title(label, fontsize=10,
                              color="green" if label == "NORMAL" else "red")
    axes[row][col].axis("off")

plt.suptitle("Augmented Training Samples (after preprocessing)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "augmented_samples.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 3.4 Compute Class Weights ──

# Design Decision: Class weights over oversampling because:
#   - Oversampling duplicates minority images → overfitting risk
#   - Class weights adjust loss function → balanced learning signal
#   - Combined with augmentation → effective imbalance handling

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_generator.classes,
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("Computed class weights:")
print(f"  Class 0 (NORMAL)    : {class_weight_dict[0]:.4f}")
print(f"  Class 1 (PNEUMONIA) : {class_weight_dict[1]:.4f}")
print(f"\nNormal class gets {class_weight_dict[0]:.2f}x weight to compensate")
print(f"for having fewer training samples (1146 vs 3300).")

---
## Section 4 — Custom CNN Model

### Architecture Design Rationale:
- **4 convolutional blocks** with increasing filters (32→64→128→256): captures hierarchical features from edges to disease patterns
- **BatchNormalization** after Conv layers: stabilises training (Ioffe & Szegedy, 2015)
- **Dropout** (0.25 conv, 0.5 dense): prevents overfitting (Srivastava et al., 2014)
- **GlobalAveragePooling2D** instead of Flatten: reduces parameters, more spatially robust
- **Sigmoid** output: binary classification probability

### Alternatives Considered:
| Choice | Selected | Alternative | Why Selected |
|---|---|---|---|
| Depth | 4 blocks | 3 or 5 blocks | 3 too shallow for lung patterns; 5 risks overfitting on ~4.4K images |
| Regularization | BatchNorm + Dropout | L2 only | Dropout + BN combo shown more effective in medical imaging |
| Pooling | GlobalAveragePooling | Flatten | GAP reduces params by ~10x, less overfitting |
| Activation | ReLU | LeakyReLU | ReLU standard; LeakyReLU marginal improvement not justified |


In [ ]:
# ── 4.1 Build Custom CNN Architecture ──

def build_custom_cnn(input_shape=(224, 224, 3)):
    """
    Custom CNN with 4 convolutional blocks.
    Designed for binary classification of chest X-rays.
    """
    model = models.Sequential(name="Custom_CNN")

    # Block 1: Low-level features (edges, simple textures)
    model.add(layers.Conv2D(32, (3, 3), activation="relu",
                            padding="same", input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Block 2: Mid-level features (corners, basic shapes)
    model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Block 3: Higher-level features (lung field patterns)
    model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Block 4: Abstract features (disease-level patterns)
    model.add(layers.Conv2D(256, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(256, (3, 3), activation="relu", padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # Classification Head
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(512, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))

    return model


model_custom = build_custom_cnn(IMG_SHAPE)
model_custom.summary()

In [ ]:
# ── 4.2 Compile Custom CNN ──

# Design Decision — Optimizer: Adam (lr=0.0001)
#   Adam chosen over SGD: adaptive per-parameter learning rates
#   0.0001 is conservative to prevent overshooting on medical images
#
# Design Decision — Primary metric: AUC
#   AUC is threshold-independent and robust for imbalanced data
#   Recall tracked separately because missed pneumonia is dangerous

model_custom.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

total_params = model_custom.count_params()
print(f"Model compiled successfully.")
print(f"Total parameters: {total_params:,}")

In [ ]:
# ── 4.3 Setup Callbacks ──

# EarlyStopping: monitors val_auc; patience=10 allows LR scheduler to try recovery
# ReduceLROnPlateau: halves LR when val_loss stalls for 5 epochs
# ModelCheckpoint: saves only the best model based on val_auc

callbacks_custom = [
    EarlyStopping(
        monitor="val_auc",
        patience=5,            # Reduced for reproducibility (original: 10)
        mode="max",
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        os.path.join(MODELS_DIR, "custom_cnn_best.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,            # Reduced for reproducibility (original: 5)
        min_lr=1e-7,
        verbose=1,
    ),
]

print("Callbacks configured:")
print("  - EarlyStopping (monitor=val_auc, patience=5)")
print("  - ModelCheckpoint (save best val_auc)")
print("  - ReduceLROnPlateau (factor=0.5, patience=3)")

In [ ]:
# ── 4.4 Train Custom CNN ──

print("=" * 60)
print("  TRAINING CUSTOM CNN")
print("=" * 60)
print(f"  Epochs     : {EPOCHS_CUSTOM} (with early stopping)")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Class wts  : Normal={class_weight_dict[0]:.2f}, Pneumonia={class_weight_dict[1]:.2f}")
print("=" * 60)

start_time = time.time()

history_custom = model_custom.fit(
    train_generator,
    epochs=EPOCHS_CUSTOM,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks_custom,
    verbose=1,
)

custom_train_time = time.time() - start_time
custom_epochs_run = len(history_custom.history["loss"])

print(f"\nTraining completed:")
print(f"  Epochs run : {custom_epochs_run} / {EPOCHS_CUSTOM}")
print(f"  Time       : {custom_train_time:.1f}s ({custom_train_time/60:.1f} min)")

In [ ]:
# ── 4.5 Plot Training History ──

def plot_training_history(history, model_name, save_path=None):
    """Plot loss, accuracy, and AUC training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Loss
    axes[0].plot(history.history["loss"], label="Train", linewidth=2)
    axes[0].plot(history.history["val_loss"], label="Validation", linewidth=2)
    axes[0].set_title("Loss", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Binary Crossentropy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(history.history["accuracy"], label="Train", linewidth=2)
    axes[1].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
    axes[1].set_title("Accuracy", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # AUC
    axes[2].plot(history.history["auc"], label="Train", linewidth=2)
    axes[2].plot(history.history["val_auc"], label="Validation", linewidth=2)
    axes[2].set_title("AUC-ROC", fontsize=13, fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("AUC")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f"{model_name} — Training History",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_training_history(
    history_custom,
    "Custom CNN",
    save_path=os.path.join(RESULTS_DIR, "custom_cnn_training.png"),
)

In [ ]:
# ── 4.6 Evaluate Custom CNN on Test Set ──

def evaluate_model(model, generator, model_name):
    """Comprehensive evaluation on test set."""
    generator.reset()

    y_pred_proba = model.predict(generator, verbose=1).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    y_true = generator.classes

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_proba)

    print(f"\n{'=' * 60}")
    print(f"  {model_name} — TEST SET EVALUATION")
    print(f"{'=' * 60}")
    print(classification_report(
        y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]
    ))
    print(f"  AUC-ROC : {auc:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"\n  Confusion Matrix Breakdown:")
    print(f"    True Negatives  (Normal ✓)     : {tn}")
    print(f"    False Positives (Normal → Pneu) : {fp}")
    print(f"    False Negatives (Pneu → Normal) : {fn}  ← missed diagnosis")
    print(f"    True Positives  (Pneumonia ✓)   : {tp}")
    print(f"{'=' * 60}")

    metrics = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "AUC-ROC": auc,
    }

    return y_true, y_pred, y_pred_proba, cm, metrics


y_true_test, y_pred_custom, y_proba_custom, cm_custom, metrics_custom = \
    evaluate_model(model_custom, test_generator, "Custom CNN")

In [ ]:
# ── 4.7 Confusion Matrix Visualization ──

def plot_confusion_matrix(cm, model_name, save_path=None):
    """Plot a labeled confusion matrix."""
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["NORMAL", "PNEUMONIA"],
        yticklabels=["NORMAL", "PNEUMONIA"],
        annot_kws={"size": 16},
    )
    plt.title(f"{model_name} — Confusion Matrix", fontsize=14, fontweight="bold")
    plt.ylabel("Actual", fontsize=12)
    plt.xlabel("Predicted", fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_confusion_matrix(
    cm_custom,
    "Custom CNN",
    save_path=os.path.join(RESULTS_DIR, "cm_custom_cnn.png"),
)

In [ ]:
# ── 4.8 Save Custom CNN & Clean GPU Memory ──

# Save metrics
pd.DataFrame([metrics_custom], index=["Custom CNN"]).to_csv(
    os.path.join(RESULTS_DIR, "custom_cnn_metrics.csv")
)

# Save model
model_custom.save(os.path.join(MODELS_DIR, "custom_cnn_final.keras"))

print("Custom CNN saved.")
print(f"  Model   : {MODELS_DIR}/custom_cnn_final.keras")
print(f"  Metrics : {RESULTS_DIR}/custom_cnn_metrics.csv")

# Store results before cleanup (needed for Section 7 comparison)
custom_results = {
    "y_pred": y_pred_custom.copy(),
    "y_proba": y_proba_custom.copy(),
    "cm": cm_custom.copy(),
    "metrics": metrics_custom.copy(),
    "train_time": custom_train_time,
    "epochs_run": custom_epochs_run,
}

# Clean GPU memory before transfer learning
del model_custom
tf.keras.backend.clear_session()
gc.collect()
print("\nGPU memory cleared for next model.")

---
## Section 5 — Transfer Learning (VGG16)

### Why Transfer Learning:
The Custom CNN achieved 0.89 AUC but struggled with Normal recall (44%). Transfer learning leverages features pre-learned from ImageNet (1.2M images) which can improve performance, especially with our limited training set (~4.4K images).

### Why VGG16:
| Model | Pros | Cons | Decision |
|---|---|---|---|
| VGG16 | Simple architecture, proven in medical imaging, clear layer structure for fine-tuning | Larger parameter count than ResNet | ✅ Selected |
| ResNet50 | Skip connections, efficient training | More complex; overkill for binary task | Considered |
| DenseNet121 | Used in CheXNet (Rajpurkar et al., 2017), efficient feature reuse | Complex architecture harder to interpret | Considered |

### Training Strategy — Two-Phase:
- **Phase 1:** Freeze all VGG16 layers → train classification head only (learn task-specific decision boundaries)
- **Phase 2:** Unfreeze last convolutional block → fine-tune with very low learning rate (adapt high-level features to X-ray domain)
- **Rationale:** Prevents catastrophic forgetting (Yosinski et al., 2014)

### Domain Gap Consideration:
ImageNet contains natural images (dogs, cars, etc.) while our task uses medical X-rays. However, early CNN layers learn universal features (edges, textures, gradients) that transfer well across domains. Only later layers need domain-specific adaptation — hence the fine-tuning phase.


In [ ]:
# ── 5.1 Build Transfer Learning Model ──

def build_transfer_model(input_shape=(224, 224, 3)):
    """
    VGG16 with frozen base + custom classification head.
    ImageNet weights provide pre-learned feature extractors.
    """
    base_model = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape,
    )

    # Freeze all base layers for Phase 1
    base_model.trainable = False

    model = models.Sequential(name="VGG16_Transfer")
    model.add(base_model)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))

    return model, base_model


model_transfer, base_model_vgg = build_transfer_model(IMG_SHAPE)
model_transfer.summary()

In [ ]:
# ── 5.2 Inspect VGG16 Layer Structure ──
# Understanding the base model layers helps justify fine-tuning decisions

print("VGG16 Layer Structure:")
print("-" * 50)
for i, layer in enumerate(base_model_vgg.layers):
    print(f"  Layer {i:2d}: {layer.name:25s} | Trainable: {layer.trainable}")

print(f"\nTotal VGG16 layers      : {len(base_model_vgg.layers)}")

# Build the model first (required by Keras 3 before count_params)
model_transfer.build(input_shape=(None, *IMG_SHAPE))

print(f"Total model parameters  : {model_transfer.count_params():,}")

trainable_params = sum(
    tf.keras.backend.count_params(w) for w in model_transfer.trainable_weights
)
print(f"Trainable parameters    : {trainable_params:,}")
print(f"Frozen parameters       : {model_transfer.count_params() - trainable_params:,}")

In [ ]:
# ── 5.3 Compile for Phase 1 ──

model_transfer.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

print("Phase 1 compiled: frozen base, training head only.")

In [ ]:
# ── 5.4 Phase 1 Callbacks ──

callbacks_transfer_p1 = [
    EarlyStopping(
        monitor="val_auc",
        patience=5,            # Reduced for reproducibility (original: 10)
        mode="max",
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        os.path.join(MODELS_DIR, "transfer_p1_best.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,            # Reduced for reproducibility (original: 5)
        min_lr=1e-7,
        verbose=1,
    ),
]

print("Phase 1 callbacks configured.")

In [ ]:
# ── 5.5 Train Phase 1: Frozen Base ──

print("=" * 60)
print("  TRANSFER LEARNING — PHASE 1: Frozen Base")
print("=" * 60)
print(f"  Epochs     : {EPOCHS_TRANSFER_P1} (with early stopping)")
print(f"  Strategy   : Only classification head is trained")
print(f"  VGG16 base : FROZEN")
print("=" * 60)

start_time_p1 = time.time()

history_transfer_p1 = model_transfer.fit(
    train_generator,
    epochs=EPOCHS_TRANSFER_P1,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks_transfer_p1,
    verbose=1,
)

p1_time = time.time() - start_time_p1
p1_epochs = len(history_transfer_p1.history["loss"])

print(f"\nPhase 1 completed:")
print(f"  Epochs run : {p1_epochs} / {EPOCHS_TRANSFER_P1}")
print(f"  Time       : {p1_time:.1f}s ({p1_time/60:.1f} min)")

In [ ]:
# ── 5.6 Unfreeze Last Block for Fine-Tuning ──

# VGG16 architecture:
#   block1 (layers 0-2)   → edges, simple textures     → KEEP FROZEN
#   block2 (layers 3-5)   → corners, simple patterns    → KEEP FROZEN
#   block3 (layers 6-9)   → medium complexity features   → KEEP FROZEN
#   block4 (layers 10-13) → higher-level features        → KEEP FROZEN
#   block5 (layers 14-18) → domain-specific features     → UNFREEZE

base_model_vgg.trainable = True

# Freeze everything BEFORE block5 (layers 0–14)
for layer in base_model_vgg.layers[:15]:
    layer.trainable = False

# Verify
print("Fine-tuning configuration:")
print("-" * 50)
for i, layer in enumerate(base_model_vgg.layers):
    status = "🟢 TRAINABLE" if layer.trainable else "🔒 FROZEN"
    print(f"  Layer {i:2d}: {layer.name:25s} | {status}")

trainable_params = sum(
    tf.keras.backend.count_params(w) for w in model_transfer.trainable_weights
)
total_params = model_transfer.count_params()
print(f"\nTrainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {total_params - trainable_params:,}")
print(f"Trainable ratio      : {trainable_params/total_params*100:.1f}%")

In [ ]:
# ── 5.7 Compile for Phase 2 (Lower Learning Rate) ──

# CRITICAL: Learning rate must be much lower for fine-tuning
# to avoid destroying pre-learned features (catastrophic forgetting)

model_transfer.compile(
    optimizer=Adam(learning_rate=1e-5),  # 10x lower than Phase 1
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

print("Phase 2 compiled with learning rate = 1e-5 (10x lower than Phase 1)")

In [ ]:
# ── 5.8 Phase 2 Callbacks ──

callbacks_transfer_p2 = [
    EarlyStopping(
        monitor="val_auc",
        patience=5,            # Reduced for reproducibility (original: 10)
        mode="max",
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        os.path.join(MODELS_DIR, "transfer_finetuned_best.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,            # Reduced for reproducibility (original: 5)
        min_lr=1e-8,
        verbose=1,
    ),
]

print("Phase 2 callbacks configured.")

In [ ]:
# ── 5.9 Train Phase 2: Fine-Tuning ──

print("=" * 60)
print("  TRANSFER LEARNING — PHASE 2: Fine-Tuning Block 5")
print("=" * 60)
print(f"  Epochs        : {EPOCHS_TRANSFER_P2} (with early stopping)")
print(f"  Learning rate : 1e-5")
print(f"  VGG16 block5  : UNFROZEN")
print("=" * 60)

start_time_p2 = time.time()

history_transfer_p2 = model_transfer.fit(
    train_generator,
    epochs=EPOCHS_TRANSFER_P2,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks_transfer_p2,
    verbose=1,
)

p2_time = time.time() - start_time_p2
p2_epochs = len(history_transfer_p2.history["loss"])
transfer_total_time = p1_time + p2_time

print(f"\nPhase 2 completed:")
print(f"  Epochs run     : {p2_epochs} / {EPOCHS_TRANSFER_P2}")
print(f"  Phase 2 time   : {p2_time:.1f}s ({p2_time/60:.1f} min)")
print(f"  Total TL time  : {transfer_total_time:.1f}s ({transfer_total_time/60:.1f} min)")

In [ ]:
# ── 5.10 Plot Combined Training History ──

def combine_histories(h1, h2):
    """Combine two Keras history objects into one."""
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]

    class CombinedHistory:
        def __init__(self, d):
            self.history = d

    return CombinedHistory(combined)


history_transfer = combine_histories(history_transfer_p1, history_transfer_p2)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
metrics_plot = [
    ("loss", "val_loss", "Loss", "Binary Crossentropy"),
    ("accuracy", "val_accuracy", "Accuracy", "Accuracy"),
    ("auc", "val_auc", "AUC-ROC", "AUC"),
]

for ax, (train_k, val_k, title, ylabel) in zip(axes, metrics_plot):
    ax.plot(history_transfer.history[train_k], label="Train", linewidth=2)
    ax.plot(history_transfer.history[val_k], label="Validation", linewidth=2)
    ax.axvline(x=p1_epochs - 1, color="gray", linestyle="--",
               alpha=0.7, label="Fine-tune start")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("VGG16 Transfer Learning — Training History (Phase 1 + Phase 2)",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "transfer_training.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 5.11 Evaluate Transfer Learning on Test Set ──

test_generator.reset()

y_true_test, y_pred_transfer, y_proba_transfer, cm_transfer, metrics_transfer = \
    evaluate_model(model_transfer, test_generator, "VGG16 Transfer Learning")

In [ ]:
# ── 5.12 Confusion Matrix ──

plot_confusion_matrix(
    cm_transfer,
    "VGG16 Transfer Learning",
    save_path=os.path.join(RESULTS_DIR, "cm_transfer.png"),
)

In [ ]:
# ── 5.13 Quick Comparison: Custom CNN vs Transfer Learning ──

print("=" * 60)
print("  QUICK COMPARISON: Custom CNN vs VGG16 Transfer")
print("=" * 60)

comparison_df = pd.DataFrame({
    "Custom CNN": custom_results["metrics"],
    "VGG16 Transfer": metrics_transfer,
}).T

print(comparison_df.round(4).to_string())

# Highlight improvement
print("\n  Improvements (Transfer vs Custom):")
for metric in metrics_transfer:
    diff = metrics_transfer[metric] - custom_results["metrics"][metric]
    arrow = "↑" if diff > 0 else "↓"
    print(f"    {metric:12s}: {diff:+.4f} {arrow}")

In [ ]:
# ── 5.14 Save Transfer Learning Results & Clean GPU ──

pd.DataFrame([metrics_transfer], index=["VGG16 Transfer"]).to_csv(
    os.path.join(RESULTS_DIR, "transfer_metrics.csv")
)

model_transfer.save(os.path.join(MODELS_DIR, "transfer_final.keras"))

print("Transfer Learning model saved.")
print(f"  Model   : {MODELS_DIR}/transfer_final.keras")
print(f"  Metrics : {RESULTS_DIR}/transfer_metrics.csv")

# Store results for Section 7
transfer_results = {
    "y_pred": y_pred_transfer.copy(),
    "y_proba": y_proba_transfer.copy(),
    "cm": cm_transfer.copy(),
    "metrics": metrics_transfer.copy(),
    "train_time": transfer_total_time,
    "epochs_run": p1_epochs + p2_epochs,
}

# Clean GPU memory for Traditional ML
del model_transfer, base_model_vgg
tf.keras.backend.clear_session()
gc.collect()
print("\nGPU memory cleared for Traditional ML section.")

---
## Section 6 — Traditional ML Baselines

### Purpose:
Establish baseline performance using traditional machine learning with hand-crafted features. This enables a direct comparison with deep learning (Criterion 1 of report).

### Key Difference — Feature Engineering:
- **Traditional ML:** Practitioner manually designs feature extractors (HOG, Haralick, LBP) based on domain knowledge
- **Deep Learning (CNN):** Network automatically learns optimal features from raw pixels

### Feature Pipeline:
| Feature Type | Method | What It Captures | Relevance to Pneumonia |
|---|---|---|---|
| Shape/Edge | HOG | Edge orientations, structural patterns | Lung field boundaries, consolidation edges |
| Texture | Haralick (GLCM) | Homogeneity, contrast, correlation | Opacity changes from fluid/infection |
| Local Texture | LBP | Micro-texture patterns | Fine-grained tissue differences |
| Statistical | Mean, Std, Skew, Kurtosis | Global intensity properties | Overall brightness changes from infection |


In [ ]:
# ── 6.1 Feature Extraction Functions ──

def extract_features(image_path, img_size=128):
    """
    Extract hand-crafted features from a single chest X-ray.
    
    Uses 128x128 (not 224x224) because:
    - HOG/LBP/Haralick don't benefit as much from high resolution
    - Reduces extraction time significantly
    - Standard practice in traditional CV pipelines
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (img_size, img_size))

    features = []

    # 1. HOG — edge/shape information
    hog_feat = hog(
        img,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        feature_vector=True,
    )
    features.extend(hog_feat)

    # 2. Haralick — GLCM texture descriptors
    haralick_feat = mahotas.features.haralick(img).mean(axis=0)
    features.extend(haralick_feat)

    # 3. LBP histogram — local texture patterns
    lbp = local_binary_pattern(img, P=24, R=3, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=26, range=(0, 26), density=True)
    features.extend(lbp_hist)

    # 4. Statistical features — global intensity
    features.extend([
        img.mean(),
        img.std(),
        float(np.median(img)),
        float(np.percentile(img, 25)),
        float(np.percentile(img, 75)),
        float(img.min()),
        float(img.max()),
        float(pd.Series(img.ravel()).skew()),
        float(pd.Series(img.ravel()).kurtosis()),
    ])

    return np.array(features, dtype=np.float64)

print("Feature extraction function defined.")
print(f"  HOG: 9 orientations, 16x16 cells, 2x2 blocks")
print(f"  Haralick: 13 GLCM-based texture features")
print(f"  LBP: 26-bin histogram (P=24, R=3)")
print(f"  Statistical: 9 global intensity features")

In [ ]:
# ── 6.2 Extract Features from Dataset ──

def extract_dataset_features(data_dir, desc="Extracting"):
    """Extract features for all images in a directory."""
    X, y = [], []
    for label, cls in enumerate(["NORMAL", "PNEUMONIA"]):
        cls_path = os.path.join(data_dir, cls)
        img_files = [
            f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))
        ]
        for img_name in tqdm(img_files, desc=f"  {desc} — {cls}"):
            img_path = os.path.join(cls_path, img_name)
            feat = extract_features(img_path)
            if feat is not None:
                X.append(feat)
                y.append(label)
    return np.array(X), np.array(y)


print("=" * 60)
print("  EXTRACTING HAND-CRAFTED FEATURES")
print("=" * 60)

start_feat_time = time.time()

print("\nTraining set:")
X_train_ml, y_train_ml = extract_dataset_features(
    os.path.join(SPLIT_DATA_DIR, "train"), desc="Train"
)

print("\nTest set:")
X_test_ml, y_test_ml = extract_dataset_features(
    os.path.join(SPLIT_DATA_DIR, "test"), desc="Test"
)

feat_time = time.time() - start_feat_time

print(f"\nFeature extraction completed in {feat_time:.1f}s ({feat_time/60:.1f} min)")
print(f"Training features shape : {X_train_ml.shape}")
print(f"Test features shape     : {X_test_ml.shape}")
print(f"Feature vector length   : {X_train_ml.shape[1]}")

In [ ]:
# ── 6.3 Scale Features ──

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ml)
X_test_scaled = scaler.transform(X_test_ml)

print("Features standardized (zero mean, unit variance).")
print(f"  Train — mean: {X_train_scaled.mean():.6f}, std: {X_train_scaled.std():.4f}")
print(f"  Test  — mean: {X_test_scaled.mean():.6f}, std: {X_test_scaled.std():.4f}")

In [ ]:
# ── 6.4 Train Traditional ML Models ──

traditional_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
        solver="lbfgs",
    ),
    "SVM (RBF)": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=SEED,
        C=10,
        gamma="scale",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
}

traditional_results = {}
traditional_predictions = {}
traditional_train_times = {}

print("=" * 60)
print("  TRAINING TRADITIONAL ML MODELS")
print("=" * 60)

for name, model in traditional_models.items():
    print(f"\n  Training {name}...")
    start_t = time.time()
    model.fit(X_train_scaled, y_train_ml)
    train_t = time.time() - start_t

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    metrics = {
        "Accuracy": accuracy_score(y_test_ml, y_pred),
        "Precision": precision_score(y_test_ml, y_pred),
        "Recall": recall_score(y_test_ml, y_pred),
        "F1-Score": f1_score(y_test_ml, y_pred),
        "AUC-ROC": roc_auc_score(y_test_ml, y_proba),
    }

    traditional_results[name] = metrics
    traditional_predictions[name] = {"y_pred": y_pred, "y_proba": y_proba}
    traditional_train_times[name] = train_t

    print(f"    Accuracy  : {metrics['Accuracy']:.4f}")
    print(f"    Precision : {metrics['Precision']:.4f}")
    print(f"    Recall    : {metrics['Recall']:.4f}")
    print(f"    F1-Score  : {metrics['F1-Score']:.4f}")
    print(f"    AUC-ROC   : {metrics['AUC-ROC']:.4f}")
    print(f"    Time      : {train_t:.2f}s")

print("\n" + "=" * 60)

In [ ]:
# ── 6.5 Feature Importance (Random Forest) ──

rf_model = traditional_models["Random Forest"]
importances = rf_model.feature_importances_

# Calculate feature group boundaries
sample_img = np.zeros((128, 128), dtype=np.uint8)
hog_len = len(hog(sample_img, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), feature_vector=True))
haralick_len = 13
lbp_len = 26
stat_len = 9

feature_groups = {
    "HOG\n(Shape/Edge)": importances[:hog_len].sum(),
    "Haralick\n(Texture)": importances[hog_len:hog_len + haralick_len].sum(),
    "LBP\n(Local Texture)": importances[hog_len + haralick_len:hog_len + haralick_len + lbp_len].sum(),
    "Statistical\n(Intensity)": importances[hog_len + haralick_len + lbp_len:].sum(),
}

feature_names = []
feature_names += [f"HOG_{i}" for i in range(hog_len)]
feature_names += [f"Haralick_{i}" for i in range(haralick_len)]
feature_names += [f"LBP_{i}" for i in range(lbp_len)]
feature_names += ["Stat_Mean", "Stat_Std", "Stat_Median", "Stat_Q25", "Stat_Q75",
                   "Stat_Min", "Stat_Max", "Stat_Skew", "Stat_Kurtosis"]

print(f"Feature names list created: {len(feature_names)} names")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]
bars = ax.barh(
    list(feature_groups.keys()),
    list(feature_groups.values()),
    color=colors,
    edgecolor="black",
)

for bar, val in zip(bars, feature_groups.values()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontweight="bold")

ax.set_xlabel("Cumulative Feature Importance", fontsize=12)
ax.set_title("Random Forest — Feature Group Importance",
             fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "feature_importance.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print("Feature importance breakdown:")
for name, val in feature_groups.items():
    print(f"  {name.replace(chr(10), ' '):25s}: {val:.4f} ({val*100:.1f}%)")

In [ ]:
# ── 6.6 Confusion Matrices for Traditional ML ──

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, preds) in enumerate(traditional_predictions.items()):
    cm = confusion_matrix(y_test_ml, preds["y_pred"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["NORMAL", "PNEUMONIA"],
        yticklabels=["NORMAL", "PNEUMONIA"],
        annot_kws={"size": 14},
        ax=axes[idx],
    )
    axes[idx].set_title(name, fontsize=13, fontweight="bold")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xlabel("Predicted")

plt.suptitle("Traditional ML — Confusion Matrices",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "cm_traditional_ml.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 6.7 SHAP Explainability for Traditional ML ──

# Install shap if not already installed
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
import shap

print("SHAP library loaded.")
print(f"SHAP version: {shap.__version__}")

# ── Compute SHAP values for Random Forest ──
# Design Decision: SHAP chosen for Traditional ML because:
#   - Provides LOCAL (per-prediction) feature explanations
#   - Grounded in Shapley values from game theory — mathematically rigorous
#   - Complements GLOBAL feature importance from Random Forest
#   - Creates meaningful contrast with Grad-CAM's SPATIAL explanations
#   - Alternative LIME was considered but rejected (stochastic, slower)

# Use TreeExplainer for Random Forest (fast, exact for tree models)
rf_model = traditional_models["Random Forest"]
explainer_rf = shap.TreeExplainer(rf_model)

print("Computing SHAP values for test set...")
start_shap_time = time.time()
shap_values_rf = explainer_rf.shap_values(X_test_scaled)
shap_time = time.time() - start_shap_time

print(f"SHAP computation completed in {shap_time:.1f}s")
print(f"SHAP values shape: {np.array(shap_values_rf).shape}")

# For binary classification, TreeExplainer returns a list of 2 arrays
# [shap_values_class_0, shap_values_class_1]
# We want class 1 (Pneumonia) explanations
if isinstance(shap_values_rf, list):
    shap_values_pneumonia = shap_values_rf[1]
else:
    shap_values_pneumonia = shap_values_rf

print(f"SHAP values for Pneumonia class shape: {shap_values_pneumonia.shape}")

In [ ]:
# ── 6.8 SHAP Summary Plot — Global Feature Importance ──

# Use the corrected 2D shap_vals from the next cell
# Recompute here to ensure correct shape
shap_vals_2d = np.array(shap_values_rf)
if shap_vals_2d.ndim == 3:
    shap_vals_2d = shap_vals_2d[:, :, 1]  # Pneumonia class

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_vals_2d,
    X_test_scaled,
    feature_names=feature_names,
    max_display=20,
    show=False,
)
plt.title("SHAP Summary — Top 20 Features (Random Forest)\nPneumonia Class",
          fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "shap_summary.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print("Interpretation:")
print("  Each dot = one test image")
print("  X-axis = SHAP value (positive → pushes toward Pneumonia)")
print("  Color = feature value (red = high, blue = low)")

In [ ]:
# ── 6.9 SHAP Individual Predictions (Force Plots) ──

# Debug: Check SHAP values shape
print(f"SHAP values type: {type(shap_values_pneumonia)}")
print(f"SHAP values shape: {np.array(shap_values_pneumonia).shape}")
print(f"Single sample shape: {np.array(shap_values_pneumonia[0]).shape}")

# Ensure SHAP values are 2D (n_samples, n_features)
shap_vals = np.array(shap_values_pneumonia)
if shap_vals.ndim == 3:
    # Some SHAP versions return (n_samples, n_features, n_classes)
    shap_vals = shap_vals[:, :, 1]  # Take pneumonia class
    print(f"Reshaped SHAP values to: {shap_vals.shape}")
elif shap_vals.ndim == 1:
    print("WARNING: SHAP values are 1D — unexpected shape")

# Find specific examples
normal_correct_idx = None
pneumonia_correct_idx = None
misclassified_idx = None

rf_predictions = traditional_predictions["Random Forest"]["y_pred"]

for i in range(len(y_test_ml)):
    if y_test_ml[i] == 0 and rf_predictions[i] == 0 and normal_correct_idx is None:
        normal_correct_idx = i
    elif y_test_ml[i] == 1 and rf_predictions[i] == 1 and pneumonia_correct_idx is None:
        pneumonia_correct_idx = i
    elif y_test_ml[i] != rf_predictions[i] and misclassified_idx is None:
        misclassified_idx = i
    
    if all(x is not None for x in [normal_correct_idx, pneumonia_correct_idx, misclassified_idx]):
        break

examples = [
    ("True Negative (Normal ✓)", normal_correct_idx),
    ("True Positive (Pneumonia ✓)", pneumonia_correct_idx),
    ("Misclassified", misclassified_idx),
]

fig, axes = plt.subplots(len(examples), 1, figsize=(14, 4 * len(examples)))

for plot_idx, (title, sample_idx) in enumerate(examples):
    if sample_idx is None:
        axes[plot_idx].text(0.5, 0.5, "No example found",
                            ha="center", va="center", fontsize=12)
        axes[plot_idx].set_title(title)
        continue

    # Get SHAP values for this sample — ensure 1D
    sample_shap = shap_vals[sample_idx].flatten()
    
    # Get top 10 most important features
    abs_shap = np.abs(sample_shap)
    top_10_idx = np.argsort(abs_shap)[-10:]
    
    top_shap_values = sample_shap[top_10_idx]
    top_feat_names = [feature_names[idx] for idx in top_10_idx.tolist()]
    
    # Color: positive SHAP = red (toward Pneumonia), negative = blue
    colors = ["#e74c3c" if v > 0 else "#3498db" for v in top_shap_values]
    
    axes[plot_idx].barh(range(len(top_feat_names)), top_shap_values,
                         color=colors, edgecolor="black")
    axes[plot_idx].set_yticks(range(len(top_feat_names)))
    axes[plot_idx].set_yticklabels(top_feat_names, fontsize=9)
    axes[plot_idx].set_xlabel("SHAP Value (impact on Pneumonia prediction)")
    
    true_label = "NORMAL" if y_test_ml[sample_idx] == 0 else "PNEUMONIA"
    pred_label = "NORMAL" if rf_predictions[sample_idx] == 0 else "PNEUMONIA"
    axes[plot_idx].set_title(
        f"{title} | True: {true_label} | Pred: {pred_label}",
        fontsize=12, fontweight="bold",
    )
    axes[plot_idx].axvline(x=0, color="black", linewidth=0.8)
    axes[plot_idx].grid(True, alpha=0.3, axis="x")

plt.suptitle("SHAP — Individual Prediction Explanations (Random Forest)",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "shap_individual.png"),
            dpi=150, bbox_inches="tight")
plt.show()

print("Red bars → Push prediction TOWARD Pneumonia")
print("Blue bars → Push prediction TOWARD Normal")

### Explainability Comparison: SHAP (Traditional ML) vs Grad-CAM (CNN)

| Aspect | SHAP (Traditional ML) | Grad-CAM (CNN) |
|---|---|---|
| **Explanation type** | Feature-based: "Haralick contrast was high" | Spatial: "Model focused on lower right lung" |
| **Granularity** | Per-feature importance values | Per-pixel importance heatmap |
| **Clinical interpretability** | Requires understanding of HOG/Haralick features | Directly shows anatomical regions — more intuitive for clinicians |
| **Mathematical basis** | Shapley values (game theory) — exact | Gradient-based — approximate |
| **Computation** | Fast for tree models (TreeExplainer) | Fast (single backward pass) |
| **Actionability** | "Texture contrast drives prediction" — abstract | "Opacity in lower lung drives prediction" — directly verifiable |

**Key Insight for Report:**

Traditional ML with SHAP tells us **which features matter** but requires feature engineering expertise to interpret. CNN with Grad-CAM tells us **where to look** in the image, which is more natural for clinicians who are trained to examine spatial patterns in X-rays.

This fundamental difference in explainability paradigm is itself a justification for using deep learning in medical imaging — the explanations are **clinically actionable** rather than statistically abstract.


---
## Section 7 — Comprehensive Comparison

This section brings together all models — Traditional ML and Deep Learning — for a complete comparative analysis across performance metrics, computational cost, and practical considerations.

In [ ]:
# ── 7.1 Full Comparison Table ──

all_results = {}

# Deep Learning models
all_results["Custom CNN"] = custom_results["metrics"].copy()
all_results["VGG16 Transfer"] = transfer_results["metrics"].copy()

# Traditional ML models
for name, metrics in traditional_results.items():
    all_results[name] = metrics.copy()

# Create DataFrame
comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df[["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]]

# Add model type column
model_types = {
    "Custom CNN": "Deep Learning",
    "VGG16 Transfer": "Deep Learning",
    "Logistic Regression": "Traditional ML",
    "SVM (RBF)": "Traditional ML",
    "Random Forest": "Traditional ML",
}
comparison_df.insert(0, "Type", [model_types[n] for n in comparison_df.index])

print("=" * 80)
print("  COMPLETE MODEL COMPARISON — TEST SET")
print("=" * 80)
print(comparison_df.round(4).to_string())
print("=" * 80)

# Save to CSV
comparison_df.to_csv(os.path.join(RESULTS_DIR, "full_comparison.csv"))
print(f"\nSaved to {RESULTS_DIR}/full_comparison.csv")

In [ ]:
# ── 7.2 Performance Comparison Bar Chart ──

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"]
model_names = comparison_df.index.tolist()

# Color by type
colors_map = {
    "Custom CNN": "#3498db",
    "VGG16 Transfer": "#2c3e50",
    "Logistic Regression": "#e67e22",
    "SVM (RBF)": "#e74c3c",
    "Random Forest": "#2ecc71",
}

fig, axes = plt.subplots(1, 5, figsize=(24, 6))

for idx, metric in enumerate(metrics_to_plot):
    values = [comparison_df.loc[m, metric] for m in model_names]
    bar_colors = [colors_map[m] for m in model_names]

    bars = axes[idx].bar(range(len(model_names)), values,
                          color=bar_colors, edgecolor="black", linewidth=0.5)
    axes[idx].set_title(metric, fontsize=13, fontweight="bold")
    axes[idx].set_ylim(0, 1.05)
    axes[idx].set_xticks(range(len(model_names)))
    axes[idx].set_xticklabels(model_names, rotation=45, ha="right", fontsize=8)
    axes[idx].grid(True, alpha=0.3, axis="y")

    for bar, val in zip(bars, values):
        axes[idx].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                       f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

plt.suptitle("Model Performance Comparison — All Metrics",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "comparison_bars.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 7.3 ROC Curve Comparison ──

plt.figure(figsize=(10, 8))

# Deep Learning models
fpr_c, tpr_c, _ = roc_curve(y_true_test, custom_results["y_proba"])
auc_c = custom_results["metrics"]["AUC-ROC"]
plt.plot(fpr_c, tpr_c, linewidth=2.5,
         label=f"Custom CNN (AUC={auc_c:.4f})", color="#3498db")

fpr_t, tpr_t, _ = roc_curve(y_true_test, transfer_results["y_proba"])
auc_t = transfer_results["metrics"]["AUC-ROC"]
plt.plot(fpr_t, tpr_t, linewidth=2.5,
         label=f"VGG16 Transfer (AUC={auc_t:.4f})", color="#2c3e50")

# Traditional ML models
trad_colors = {"Logistic Regression": "#e67e22", "SVM (RBF)": "#e74c3c",
               "Random Forest": "#2ecc71"}
for name, preds in traditional_predictions.items():
    fpr, tpr, _ = roc_curve(y_test_ml, preds["y_proba"])
    auc_val = traditional_results[name]["AUC-ROC"]
    plt.plot(fpr, tpr, linewidth=2,
             label=f"{name} (AUC={auc_val:.4f})", color=trad_colors[name])

# Random baseline
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC=0.5)")

plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve Comparison — All Models", fontsize=16, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 7.4 Computational Cost Comparison ──

time_data = {
    "Custom CNN": custom_results["train_time"],
    "VGG16 Transfer": transfer_results["train_time"],
}
for name, t in traditional_train_times.items():
    time_data[name] = t

# Add feature extraction time split across traditional models
feat_time_per_model = feat_time / len(traditional_models)

time_df = pd.DataFrame({
    "Model": list(time_data.keys()),
    "Training Time (s)": list(time_data.values()),
    "Training Time (min)": [t / 60 for t in time_data.values()],
})
time_df["Type"] = [model_types[n] for n in time_df["Model"]]

print("=" * 60)
print("  COMPUTATIONAL COST COMPARISON")
print("=" * 60)
print(time_df.to_string(index=False))
print(f"\n  Feature extraction time (Traditional ML): {feat_time:.1f}s ({feat_time/60:.1f} min)")
print("=" * 60)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [colors_map[m] for m in time_df["Model"]]
bars = ax.barh(time_df["Model"], time_df["Training Time (min)"],
               color=bar_colors, edgecolor="black")

for bar, val in zip(bars, time_df["Training Time (min)"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f} min", va="center", fontweight="bold")

ax.set_xlabel("Training Time (minutes)", fontsize=12)
ax.set_title("Computational Cost Comparison",
             fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_time.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 7.5 All Confusion Matrices Side by Side ──

fig, axes = plt.subplots(1, 5, figsize=(28, 5))

all_cms = {
    "Custom CNN": custom_results["cm"],
    "VGG16 Transfer": transfer_results["cm"],
}
for name, preds in traditional_predictions.items():
    all_cms[name] = confusion_matrix(y_test_ml, preds["y_pred"])

for idx, (name, cm) in enumerate(all_cms.items()):
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["NOR", "PNEU"],
        yticklabels=["NOR", "PNEU"],
        annot_kws={"size": 13},
        ax=axes[idx],
    )
    axes[idx].set_title(name, fontsize=11, fontweight="bold")
    axes[idx].set_ylabel("Actual" if idx == 0 else "")
    axes[idx].set_xlabel("Predicted")

plt.suptitle("Confusion Matrices — All Models",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "all_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

### Section 7 — Key Findings Summary

**Performance Ranking** (by AUC-ROC):
1. VGG16 Transfer Learning — best overall
2. Custom CNN — high recall but poor precision
3. Traditional ML models — competitive but limited by hand-crafted features

**Critical Observations for Report:**

| Observation | Implication |
|---|---|
| Transfer learning significantly outperforms custom CNN | Pre-learned features from ImageNet transfer well to medical imaging despite domain gap |
| Custom CNN has very high recall (98%) but low precision | Model is overly aggressive — predicts pneumonia too often. Smaller dataset limits learning balanced features |
| Traditional ML achieves reasonable performance | Hand-crafted features capture useful patterns, but ceiling is lower than learned features |
| DL training takes 10-50x longer than traditional ML | Significant computational cost tradeoff for better performance |
| False negatives (missed pneumonia) are lowest in CNN models | DL models are more clinically safe for screening |

**For the report (Criterion 1):** The comparison demonstrates that deep learning's automatic feature learning provides a clear advantage over manual feature engineering, particularly for capturing subtle spatial patterns in medical images. However, this comes at a significant computational cost.


---
## Section 8 — Grad-CAM Interpretability

### Purpose:
Visualise which regions of the X-ray the CNN focuses on when making predictions. This addresses:
- **Clinical trust:** Do the model's attention regions align with known pneumonia indicators?
- **Model debugging:** Are there cases where the model focuses on irrelevant features?
- **Report Criterion 2:** Interpretability as a design consideration
- **Report Criterion 3:** Impact on clinical adoption and trust

### Method:
Grad-CAM (Gradient-weighted Class Activation Mapping) computes the gradient of the predicted class score with respect to the final convolutional layer's feature maps. This produces a heatmap highlighting regions most influential to the prediction (Selvaraju et al., 2017).

In [ ]:
# ── 8.1 Reload Best Model for Grad-CAM ──

model_gradcam = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, "transfer_final.keras")
)

# IMPORTANT: Call model once with dummy input to build graph
# This is required for Keras 3 to define input/output tensors
dummy_input = np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
_ = model_gradcam(dummy_input)

print("Model loaded and built for Grad-CAM analysis.")
print(f"Model name: {model_gradcam.name}")
print(f"Input shape: {model_gradcam.input_shape}")
print(f"Output shape: {model_gradcam.output_shape}")

In [ ]:
# ── 8.2 Identify the Target Convolutional Layer ──

# The base VGG16 is the first layer of our Sequential model
# We need the last conv layer INSIDE VGG16 for Grad-CAM

base_model = model_gradcam.layers[0]  # VGG16 base

print("Layers in VGG16 base model:")
print("-" * 50)
for i, layer in enumerate(base_model.layers):
    layer_type = layer.__class__.__name__
    print(f"  {i:2d}: {layer.name:25s} | {layer_type}")

# The last convolutional layer in VGG16 is 'block5_conv3'
GRAD_CAM_LAYER = "block5_conv3"
print(f"\nTarget layer for Grad-CAM: {GRAD_CAM_LAYER}")

In [ ]:
# ── 8.3 Grad-CAM Implementation (Tape-Only Approach) ──

# This approach avoids building Keras sub-models which cause issues
# on Keras 3 + Apple Metal. Instead, we manually forward through
# layers and capture intermediate outputs with GradientTape.

def gradcam_tape_only(model, img_array, target_layer_name):
    """
    Grad-CAM implementation using only GradientTape.
    Compatible with Keras 3 and CUDA/CPU backends.
    """
    img_tensor = tf.cast(img_array, tf.float32)

    # Check if target layer is inside a nested model (e.g. VGG16 base)
    is_nested = False
    base_model = None
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for sub_layer in layer.layers:
                if sub_layer.name == target_layer_name:
                    is_nested = True
                    base_model = layer
                    break

    if is_nested:
        # VGG16 transfer model — target layer is inside base model
        extractor = Model(
            inputs=base_model.input,
            outputs=base_model.get_layer(target_layer_name).output,
        )

        with tf.GradientTape() as tape:
            conv_output = extractor(img_tensor)
            tape.watch(conv_output)

            x = base_model(img_tensor)
            for layer in model.layers[1:]:
                x = layer(x)
            prediction = x[:, 0]

        grads = tape.gradient(prediction, conv_output)

    else:
        # Flat Sequential model (Custom CNN)
        with tf.GradientTape() as tape:
            x = img_tensor
            conv_output = None

            for layer in model.layers:
                x = layer(x)
                if layer.name == target_layer_name:
                    conv_output = x
                    tape.watch(conv_output)

            prediction = x[:, 0]

        grads = tape.gradient(prediction, conv_output)

    # Process heatmap
    if grads is None or conv_output is None:
        print(f"  Warning: Using activation-only heatmap for {target_layer_name}")
        if conv_output is not None:
            heatmap = tf.reduce_mean(conv_output[0], axis=-1)
        else:
            return np.zeros((14, 14))
    else:
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_output_val = conv_output[0]
        heatmap = conv_output_val @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def preprocess_image_for_gradcam(image_path, img_size=224):
    """Load and preprocess a single image for Grad-CAM."""
    img = tf.keras.preprocessing.image.load_img(
        image_path, target_size=(img_size, img_size)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    return img_array, np.expand_dims(img_array, axis=0)


# ── Quick test ──
test_img_path = None
for cls in ["NORMAL", "PNEUMONIA"]:
    cls_path = os.path.join(SPLIT_DATA_DIR, "test", cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    if imgs:
        test_img_path = os.path.join(cls_path, imgs[0])
        break

img_display_test, img_batch_test = preprocess_image_for_gradcam(test_img_path)

print("Testing Grad-CAM on VGG16 Transfer...")
hm_vgg = gradcam_tape_only(model_gradcam, img_batch_test, GRAD_CAM_LAYER)
print(f"  VGG16 heatmap shape: {hm_vgg.shape}")
print("\nGrad-CAM functions defined and tested.")

In [ ]:
# ── 8.4 Find Example Images from Each Prediction Category ──

def find_example_images(test_dir, model, generator, n_per_category=2):
    """Find example images for each prediction category."""
    generator.reset()
    y_proba = model.predict(generator, verbose=0).flatten()
    y_pred = (y_proba > 0.5).astype(int)
    y_true = generator.classes
    filenames = generator.filenames

    categories = {
        "True Positive (Pneumonia correct)": [],
        "True Negative (Normal correct)": [],
        "False Positive (Normal as Pneu)": [],
        "False Negative (Pneu missed)": [],
    }

    for i in range(len(y_true)):
        if y_true[i] == 1 and y_pred[i] == 1:
            cat = "True Positive (Pneumonia correct)"
        elif y_true[i] == 0 and y_pred[i] == 0:
            cat = "True Negative (Normal correct)"
        elif y_true[i] == 0 and y_pred[i] == 1:
            cat = "False Positive (Normal as Pneu)"
        else:
            cat = "False Negative (Pneu missed)"

        if len(categories[cat]) < n_per_category:
            categories[cat].append({
                "filepath": os.path.join(test_dir, filenames[i]),
                "true_label": "PNEUMONIA" if y_true[i] == 1 else "NORMAL",
                "pred_label": "PNEUMONIA" if y_pred[i] == 1 else "NORMAL",
                "confidence": float(y_proba[i]),
                "category": cat,
            })

    return categories


test_generator.reset()
example_images = find_example_images(
    os.path.join(SPLIT_DATA_DIR, "test"),
    model_gradcam,
    test_generator,
    n_per_category=2,
)

all_examples = []
for cat, examples in example_images.items():
    print(f"\n{cat} — {len(examples)} examples")
    for ex in examples:
        print(f"  {os.path.basename(ex['filepath'])} | "
              f"True: {ex['true_label']} | Pred: {ex['pred_label']} | "
              f"Conf: {ex['confidence']:.3f}")
        all_examples.append(ex)

In [ ]:
# ── 8.5 Generate Grad-CAM Visualizations ──

n_examples = len(all_examples)
fig, axes = plt.subplots(n_examples, 3, figsize=(14, 4.5 * n_examples))

if n_examples == 1:
    axes = [axes]

for idx, ex in enumerate(all_examples):
    img_display, img_batch = preprocess_image_for_gradcam(ex["filepath"])

    heatmap = gradcam_tape_only(model_gradcam, img_batch, GRAD_CAM_LAYER)

    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay = np.clip(heatmap_colored * 0.4 + img_display * 0.6, 0, 1)

    title_color = "green" if "True" in ex["category"] else "red"

    axes[idx][0].imshow(img_display)
    axes[idx][0].set_title(
        f"{ex['category']}\nTrue: {ex['true_label']} | Pred: {ex['pred_label']}",
        fontsize=9, color=title_color, fontweight="bold",
    )
    axes[idx][0].axis("off")

    axes[idx][1].imshow(heatmap_resized, cmap="jet")
    axes[idx][1].set_title("Grad-CAM Heatmap", fontsize=9)
    axes[idx][1].axis("off")

    axes[idx][2].imshow(overlay)
    axes[idx][2].set_title(f"Overlay (conf: {ex['confidence']:.3f})", fontsize=9)
    axes[idx][2].axis("off")

plt.suptitle("Grad-CAM Analysis — VGG16 Transfer Learning Model",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "gradcam_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 8.6 Reload Custom CNN for Comparison ──

model_custom_reload = tf.keras.models.load_model(
    os.path.join(MODELS_DIR, "custom_cnn_final.keras")
)

# Build graph
dummy_input = np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
_ = model_custom_reload(dummy_input)

custom_conv_layers = [
    layer.name for layer in model_custom_reload.layers
    if isinstance(layer, layers.Conv2D)
]
CUSTOM_LAST_CONV = custom_conv_layers[-1]

# Test Grad-CAM on Custom CNN
print(f"Custom CNN loaded. Last conv layer: {CUSTOM_LAST_CONV}")
print("\nTesting Grad-CAM on Custom CNN...")
hm_custom_test = gradcam_tape_only(model_custom_reload, img_batch_test, CUSTOM_LAST_CONV)
print(f"  Custom CNN heatmap shape: {hm_custom_test.shape}")

In [ ]:
# ── 8.7 Grad-CAM Comparison: Custom CNN vs VGG16 ──

sample_paths = []
for cls in ["NORMAL", "PNEUMONIA"]:
    cls_path = os.path.join(SPLIT_DATA_DIR, "test", cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    selected = random.sample(imgs, 2)
    for img_name in selected:
        sample_paths.append({
            "path": os.path.join(cls_path, img_name),
            "label": cls,
        })

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(18, 4.5 * len(sample_paths)))

for idx, sample in enumerate(sample_paths):
    img_display, img_batch = preprocess_image_for_gradcam(sample["path"])

    # VGG16 Grad-CAM
    heatmap_vgg = gradcam_tape_only(model_gradcam, img_batch, GRAD_CAM_LAYER)
    heatmap_vgg_resized = cv2.resize(heatmap_vgg, (IMG_SIZE, IMG_SIZE))
    heatmap_vgg_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_vgg_resized), cv2.COLORMAP_JET
    )
    heatmap_vgg_colored = cv2.cvtColor(heatmap_vgg_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay_vgg = np.clip(heatmap_vgg_colored * 0.4 + img_display * 0.6, 0, 1)

    # Custom CNN Grad-CAM
    heatmap_custom = gradcam_tape_only(model_custom_reload, img_batch, CUSTOM_LAST_CONV)
    heatmap_custom_resized = cv2.resize(heatmap_custom, (IMG_SIZE, IMG_SIZE))
    heatmap_custom_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_custom_resized), cv2.COLORMAP_JET
    )
    heatmap_custom_colored = cv2.cvtColor(heatmap_custom_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay_custom = np.clip(heatmap_custom_colored * 0.4 + img_display * 0.6, 0, 1)

    # Predictions
    pred_vgg = float(model_gradcam.predict(img_batch, verbose=0)[0][0])
    pred_custom = float(model_custom_reload.predict(img_batch, verbose=0)[0][0])

    axes[idx][0].imshow(img_display)
    axes[idx][0].set_title(f"Original\nTrue: {sample['label']}", fontsize=10)
    axes[idx][0].axis("off")

    axes[idx][1].imshow(overlay_custom)
    pred_label_c = "PNEU" if pred_custom > 0.5 else "NOR"
    axes[idx][1].set_title(f"Custom CNN\nPred: {pred_label_c} ({pred_custom:.3f})", fontsize=10)
    axes[idx][1].axis("off")

    axes[idx][2].imshow(overlay_vgg)
    pred_label_v = "PNEU" if pred_vgg > 0.5 else "NOR"
    axes[idx][2].set_title(f"VGG16 Transfer\nPred: {pred_label_v} ({pred_vgg:.3f})", fontsize=10)
    axes[idx][2].axis("off")

    diff_heatmap = np.abs(heatmap_vgg_resized - heatmap_custom_resized)
    axes[idx][3].imshow(diff_heatmap, cmap="hot")
    axes[idx][3].set_title("Attention Difference\n(VGG16 vs Custom)", fontsize=10)
    axes[idx][3].axis("off")

plt.suptitle("Grad-CAM Comparison: Custom CNN vs VGG16 Transfer Learning",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "gradcam_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 8.8 Visual Comparison: Custom CNN vs VGG16 Grad-CAM ──

sample_paths = []
for cls in ["NORMAL", "PNEUMONIA"]:
    cls_path = os.path.join(SPLIT_DATA_DIR, "test", cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith((".jpeg", ".jpg", ".png"))]
    selected = random.sample(imgs, 2)
    for img_name in selected:
        sample_paths.append({
            "path": os.path.join(cls_path, img_name),
            "label": cls,
        })

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(18, 4.5 * len(sample_paths)))

for idx, sample in enumerate(sample_paths):
    img_display, img_batch = preprocess_image_for_gradcam(sample["path"])

    # VGG16 Grad-CAM
    heatmap_vgg = gradcam_tape_only(model_gradcam, img_batch, GRAD_CAM_LAYER)
    heatmap_vgg_resized = cv2.resize(heatmap_vgg, (IMG_SIZE, IMG_SIZE))
    heatmap_vgg_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_vgg_resized), cv2.COLORMAP_JET
    )
    heatmap_vgg_colored = cv2.cvtColor(heatmap_vgg_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay_vgg = np.clip(heatmap_vgg_colored * 0.4 + img_display * 0.6, 0, 1)

    # Custom CNN Grad-CAM
    heatmap_custom = gradcam_tape_only(model_custom_reload, img_batch, CUSTOM_LAST_CONV)
    heatmap_custom_resized = cv2.resize(heatmap_custom, (IMG_SIZE, IMG_SIZE))
    heatmap_custom_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_custom_resized), cv2.COLORMAP_JET
    )
    heatmap_custom_colored = cv2.cvtColor(heatmap_custom_colored, cv2.COLOR_BGR2RGB) / 255.0
    overlay_custom = np.clip(heatmap_custom_colored * 0.4 + img_display * 0.6, 0, 1)

    # Predictions
    pred_vgg = float(model_gradcam.predict(img_batch, verbose=0)[0][0])
    pred_custom = float(model_custom_reload.predict(img_batch, verbose=0)[0][0])

    axes[idx][0].imshow(img_display)
    axes[idx][0].set_title(f"Original\nTrue: {sample['label']}", fontsize=10)
    axes[idx][0].axis("off")

    axes[idx][1].imshow(overlay_custom)
    pred_label_c = "PNEU" if pred_custom > 0.5 else "NOR"
    axes[idx][1].set_title(f"Custom CNN\nPred: {pred_label_c} ({pred_custom:.3f})", fontsize=10)
    axes[idx][1].axis("off")

    axes[idx][2].imshow(overlay_vgg)
    pred_label_v = "PNEU" if pred_vgg > 0.5 else "NOR"
    axes[idx][2].set_title(f"VGG16 Transfer\nPred: {pred_label_v} ({pred_vgg:.3f})", fontsize=10)
    axes[idx][2].axis("off")

    diff_heatmap = np.abs(heatmap_vgg_resized - heatmap_custom_resized)
    axes[idx][3].imshow(diff_heatmap, cmap="hot")
    axes[idx][3].set_title("Attention Difference\n(VGG16 vs Custom)", fontsize=10)
    axes[idx][3].axis("off")

plt.suptitle("Grad-CAM Comparison: Custom CNN vs VGG16 Transfer Learning",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "gradcam_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

---
## Section 9 — Conclusions & Summary

In [ ]:
# ── 9.1 Final Results Summary ──

print("=" * 80)
print("  FINAL RESULTS SUMMARY")
print("=" * 80)

# Reload comparison data
final_df = pd.read_csv(os.path.join(RESULTS_DIR, "full_comparison.csv"), index_col=0)
print("\nPerformance Metrics (Test Set):")
print(final_df.round(4).to_string())

# Best model
best_model = final_df["AUC-ROC"].astype(float).idxmax()
best_auc = final_df.loc[best_model, "AUC-ROC"]
print(f"\n🏆 Best Model: {best_model} (AUC-ROC = {best_auc})")

# Training times
print(f"\nTraining Times:")
print(f"  Custom CNN         : {custom_results['train_time']/60:.1f} min ({custom_results['epochs_run']} epochs)")
print(f"  VGG16 Transfer     : {transfer_results['train_time']/60:.1f} min ({transfer_results['epochs_run']} epochs)")
for name, t in traditional_train_times.items():
    print(f"  {name:21s}: {t:.1f}s")
print(f"  Feature extraction : {feat_time:.1f}s")

print("\n" + "=" * 80)

In [ ]:
# ── 9.2 Summary of All Saved Outputs ──

print("=" * 60)
print("  SAVED OUTPUTS")
print("=" * 60)

print("\n📁 Models:")
for f in os.listdir(MODELS_DIR):
    filepath = os.path.join(MODELS_DIR, f)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  {f:40s} ({size_mb:.1f} MB)")

print("\n📁 Results:")
for f in sorted(os.listdir(RESULTS_DIR)):
    filepath = os.path.join(RESULTS_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:40s} ({size_kb:.1f} KB)")

print("\n" + "=" * 60)

### Project Summary

**Problem:** Binary classification of chest X-rays (Normal vs Pneumonia)

**Dataset:** Kermany et al. (2018) — 5,232 training + 624 test images

**Models Built:**
1. **Custom CNN** (4-block architecture, trained from scratch)
2. **VGG16 Transfer Learning** (2-phase: frozen base → fine-tuning)
3. **Traditional ML** (Logistic Regression, SVM, Random Forest with HOG/Haralick/LBP features)

**Key Findings:**

| Finding | Evidence |
|---|---|
| Transfer learning outperforms all other approaches | AUC: 0.97 vs 0.89 (Custom CNN) vs 0.93 (best Traditional ML) |
| Deep learning provides balanced predictions | VGG16: 89% Normal recall + 96% Pneumonia recall |
| Traditional ML has high recall but very low precision | All 3 models predict ~99.5% as Pneumonia — poor discrimination |
| Custom CNN overfits toward Pneumonia class | 98% Pneumonia recall but only 44% Normal recall |
| Grad-CAM shows VGG16 focuses on clinically relevant regions | Attention maps highlight lung fields, not irrelevant artifacts |
| DL training is 100-1000x more expensive than Traditional ML | 45-50 min (DL) vs seconds (Traditional ML) |

**Design Decisions That Mattered Most:**
1. Transfer learning with pre-trained ImageNet features
2. Two-phase training (frozen → fine-tuned)
3. Class weights to handle 2.88:1 imbalance
4. AUC as primary metric instead of accuracy
5. Data augmentation with clinically appropriate transformations

**Limitations:**
- Single-centre dataset — may not generalise to other hospitals/X-ray machines
- Binary classification only — does not distinguish bacterial vs viral pneumonia
- Small dataset (~5K images) limits custom CNN performance
- No external validation on a separate dataset

**All results, plots, and models saved in `./results/` and `./models/` directories.**
